# Edge AI — Crop Disease Detection Training
**Project:** Edge AI-Based Crop Disease Detection and Automated Irrigation System  
**Author:** George Obinna Oguejiofor | KMUTT  

Trains **MobileNetV2** on **PlantVillage** (15 classes) and exports an **int8 TFLite** model
ready for embedded deployment (ESP32-S3, Raspberry Pi).

### Run order
1. Cell 1a — fix deps & restart *(run once, skip on subsequent sessions)*
2. Cell 1b — set save directory
3. Cell 2 — Kaggle auth & dataset download
4. Cell 3–11 — train, evaluate, convert, download

> **Before running:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── 1a. Fix dependencies & restart ────────────────────────────────────────────
# Run ONCE on a fresh runtime. The runtime restarts automatically.
# Skip this cell if you already ran it this session.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--upgrade', 'protobuf', 'tensorflow-metadata',
                'importlib_resources', 'seaborn'], check=True)

print('✓ Dependencies fixed. Restarting runtime...')
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── 1b. Set save directory ────────────────────────────────────────────────
# Run AFTER the runtime restarts (skip Cell 1a the second time).
import os
SAVE_DIR = '/content/models'
os.makedirs(SAVE_DIR, exist_ok=True)
print('✓ Save directory:', SAVE_DIR)

In [ ]:
# ── 2. Kaggle auth & dataset download ─────────────────────────────────────────
# Requires KAGGLE_API_TOKEN in Colab Secrets:
#   Left sidebar → 🔑 Secrets → + Add new secret
#   Name: KAGGLE_API_TOKEN   Value: your token from kaggle.com → Settings → API
import os, sys, subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], check=True)

try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if not token:
        raise ValueError('empty')
    print('✓ Using Kaggle token from Colab Secrets')
except Exception:
    import getpass
    token = getpass.getpass('Enter Kaggle API token (KGAT_...): ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)
os.chmod('/root/.kaggle/access_token', 0o600)
print('✓ Kaggle token configured')

DATA_DIR = '/content/plantdisease'
if not os.path.exists(DATA_DIR):
    print('Downloading PlantVillage...')
    os.system('kaggle datasets download -d emmarex/plantdisease -p /content/ --quiet')
    os.system('unzip -q /content/plantdisease.zip -d /content/plantdisease/')
    print('✓ Downloaded and extracted')
else:
    print('✓ PlantVillage already on disk')

import pathlib
root = pathlib.Path(DATA_DIR)
img_dir = root
for candidate in sorted(root.rglob('*')):
    if candidate.is_dir():
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        if len(subdirs) >= 10:
            img_dir = candidate
            break

CLASS_NAMES = sorted([d.name for d in img_dir.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
print(f'✓ Image directory : {img_dir}')
print(f'✓ Classes found   : {NUM_CLASSES}')

In [ ]:
# ── 3. Imports, config & load dataset ──────────────────────────────────────────
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

IMG_SIZE   = 128
BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE
SEED       = 42

# img_dir, CLASS_NAMES, NUM_CLASSES set in Cell 2
ds_train_raw = tf.keras.utils.image_dataset_from_directory(
    str(img_dir),
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None,
    label_mode='int',
    class_names=CLASS_NAMES,
)

ds_rest_raw = tf.keras.utils.image_dataset_from_directory(
    str(img_dir),
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None,
    label_mode='int',
    class_names=CLASS_NAMES,
)

n_rest = int(ds_rest_raw.cardinality())
n_val  = n_rest // 2
ds_val_raw  = ds_rest_raw.take(n_val)
ds_test_raw = ds_rest_raw.skip(n_val)

n_train = int(ds_train_raw.cardinality())
print(f'✓ Train  : {n_train} samples')
print(f'✓ Val    : {n_val} samples')
print(f'✓ Test   : {n_rest - n_val} samples')
print(f'✓ Classes: {NUM_CLASSES}')

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (image, label) in zip(axes.flat, ds_train_raw.take(10)):
    ax.imshow(image.numpy().astype('uint8'))
    ax.set_title(CLASS_NAMES[int(label)].replace('_', '\n'), fontsize=7)
    ax.axis('off')
plt.suptitle('PlantVillage — sample images', fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/sample_images.png', dpi=100)
plt.show()
print('✓ Dataset ready')

In [ ]:
# ── 4. Preprocessing & augmentation ───────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    return image, label

ds_train_ready = (
    ds_train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .shuffle(2000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_val_ready = (
    ds_val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_test_ready = (
    ds_test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
print('✓ Datasets ready')

In [ ]:
# ── 5. Build model ─────────────────────────────────────────────────────────────
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x       = base(x, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model   = tf.keras.Model(inputs, outputs)

model.summary()

In [ ]:
# ── 6. Phase 1 — Transfer learning (frozen base, 15 epochs) ────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(f'{SAVE_DIR}/phase1_best.keras',
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
]

print('=== Phase 1: Transfer Learning (frozen base) ===')
h1 = model.fit(ds_train_ready, validation_data=ds_val_ready,
               epochs=15, callbacks=callbacks_p1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ['accuracy', 'loss']):
    ax.plot(h1.history[m], label=f'train {m}')
    ax.plot(h1.history[f'val_{m}'], label=f'val {m}')
    ax.legend(); ax.set_title(f'Phase 1 — {m}')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/phase1_curve.png', dpi=100)
plt.show()

In [ ]:
# ── 7. Phase 2 — Fine-tuning (top 30 layers unfrozen, 10 epochs) ───────────────
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(f'{SAVE_DIR}/phase2_best.keras',
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
]

print('=== Phase 2: Fine-tuning (top 30 layers unfrozen) ===')
h2 = model.fit(ds_train_ready, validation_data=ds_val_ready,
               epochs=10, callbacks=callbacks_p2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ['accuracy', 'loss']):
    ax.plot(h2.history[m], label=f'train {m}')
    ax.plot(h2.history[f'val_{m}'], label=f'val {m}')
    ax.legend(); ax.set_title(f'Phase 2 — {m}')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/phase2_curve.png', dpi=100)
plt.show()

In [ ]:
# ── 8. Evaluate on test set ──────────────────────────────────────────────────
print('=== Test Set Evaluation ===')
loss, acc = model.evaluate(ds_test_ready)
print(f'Test Accuracy : {acc:.4f}')
print(f'Test Loss     : {loss:.4f}')

y_true, y_pred = [], []
for images, labels in ds_test_ready:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)
with open(f'{SAVE_DIR}/classification_report.txt', 'w') as f:
    f.write(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(22, 20))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — Accuracy {acc:.1%}')
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# ── 9. Save metadata & convert to int8 TFLite ────────────────────────────────
# Uses from_keras_model to avoid protobuf issues with model.save().
import json

meta = {
    'class_names'   : CLASS_NAMES,
    'img_size'      : IMG_SIZE,
    'num_classes'   : NUM_CLASSES,
    'test_accuracy' : round(float(acc), 4),
}
meta_path = f'{SAVE_DIR}/model_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✓ Metadata saved: {meta_path}')

# Representative dataset for int8 calibration (200 samples)
ds_repr = ds_train_ready.unbatch().batch(1).take(200)
def representative_dataset():
    for img, _ in ds_repr:
        yield [tf.cast(img, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.uint8
converter.inference_output_type = tf.uint8
int8_model = converter.convert()

int8_path = f'{SAVE_DIR}/mobilenetv2_int8.tflite'
with open(int8_path, 'wb') as f:
    f.write(int8_model)
size_kb = len(int8_model) // 1024
print(f'✓ int8 TFLite saved: {int8_path} ({size_kb} KB)')
print('✓ Ready for ESP32-S3 / Raspberry Pi deployment.')

In [ ]:
# ── 10. Latency benchmark (Colab CPU backend) ────────────────────────────────
import time as _time

interpreter = tf.lite.Interpreter(model_path=int8_path)
interpreter.allocate_tensors()
inp_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]

dummy = np.random.randint(0, 255, (1, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
times = []
for _ in range(50):
    interpreter.set_tensor(inp_det['index'], dummy)
    t0 = _time.perf_counter()
    interpreter.invoke()
    times.append(_time.perf_counter() - t0)

mean_ms = np.mean(times) * 1000
print(f'int8 TFLite latency (Colab CPU backend): {mean_ms:.1f} ms')
print(f'Estimated on ESP32-S3 (~10x slower)    : {mean_ms*10:.0f} ms')
print(f'Estimated on Raspberry Pi 4 (~2x slower): {mean_ms*2:.0f} ms')

In [ ]:
# ── 11. Download model files to your machine ────────────────────────────────
# Downloads mobilenetv2_int8.tflite and model_meta.json to your browser's
# Downloads folder. Move them into: disease_detection/models/
from google.colab import files

print('Downloading...')
files.download(int8_path)
files.download(meta_path)
print(f'✓ {int8_path}')
print(f'✓ {meta_path}')
print('Done. Move these files to disease_detection/models/')

## After downloading

Move the two downloaded files into your local project:
```
disease_detection/models/mobilenetv2_int8.tflite
disease_detection/models/model_meta.json
```

**Model summary** (from the run that produced the downloaded files):
- Dataset: PlantVillage via Kaggle (`emmarex/plantdisease`)
- Classes: 15 (Pepper, Potato, Tomato diseases + healthy)
- Input: 128×128 RGB uint8
- Quantisation: int8 (weights + activations)
- Test accuracy: **96.56 %**
- Model size: ~2.7 MB